# 19 — Data Persistence: SQLite, SQL, and ORM Concepts

Goal: store and query data safely using SQLite (stdlib) and understand ORM tradeoffs.

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
python -m pip install -U sqlalchemy alembic
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: SQLite basics (stdlib `sqlite3`)

SQLite is a file-based SQL database.
Great for:
- local apps
- prototypes
- small services

Not great for:
- high write concurrency across many machines

In [ ]:

import sqlite3

con = sqlite3.connect(":memory:")
cur = con.cursor()

cur.execute("CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT, active INTEGER)")
cur.execute("INSERT INTO users (name, active) VALUES (?, ?)", ("Ada", 1))
cur.execute("INSERT INTO users (name, active) VALUES (?, ?)", ("Grace", 0))
con.commit()

cur.execute("SELECT id, name, active FROM users WHERE active = ?", (1,))
print(cur.fetchall())

con.close()


## 2.
L2: Transactions and context managers

Use transactions to keep data consistent.

In [ ]:

import sqlite3

with sqlite3.connect(":memory:") as con:
    con.execute("CREATE TABLE items (id INTEGER PRIMARY KEY, v INTEGER)")
    # inside this block, commit happens automatically if no exception
    con.execute("INSERT INTO items (v) VALUES (?)", (1,))
    con.execute("INSERT INTO items (v) VALUES (?)", (2,))
    rows = con.execute("SELECT v FROM items ORDER BY v").fetchall()

print(rows)


## 3.
L3: SQL injection (why parameterized queries matter)

Never format user input into SQL strings.
Always use placeholders (`?` for sqlite).

In [ ]:

import sqlite3

with sqlite3.connect(":memory:") as con:
    con.execute("CREATE TABLE demo (name TEXT)")
    user_input = "Ada'); DROP TABLE demo; --"

    # safe:
    con.execute("INSERT INTO demo (name) VALUES (?)", (user_input,))
    print(con.execute("SELECT name FROM demo").fetchall())


## 4.
L4: Row factories (dict-like access)

You can return rows as dict-like objects for readability.

In [ ]:

import sqlite3

con = sqlite3.connect(":memory:")
con.row_factory = sqlite3.Row
con.execute("CREATE TABLE t (a INTEGER, b TEXT)")
con.execute("INSERT INTO t VALUES (?,?)", (1, "x"))
row = con.execute("SELECT a,b FROM t").fetchone()
print(row["a"], row["b"], dict(row))
con.close()


## 5.
L5: ORMs (SQLAlchemy) — why/when

ORMs can:
- map tables to classes
- manage relationships
- help with migrations

They also add:
- complexity
- performance pitfalls if you don’t understand generated SQL

Rule: learn SQL first, then use an ORM deliberately.

## 6.
L6: Exercises

1. Create a table `events(id, ts, payload)` and insert 10 rows.
2. Query rows with `WHERE` + `ORDER BY` + `LIMIT`.
3. Explain, in your own words, what a transaction guarantees.

## 7.
L7: Constraints and indexes (conceptual)

- constraints enforce correctness (PRIMARY KEY, UNIQUE, NOT NULL, CHECK, FOREIGN KEY)
- indexes speed up reads at the cost of write overhead

In SQLite, enable foreign keys:
```sql
PRAGMA foreign_keys = ON;
```

## 8.
L8: Migrations (schema changes)

In real apps, schemas evolve.
Migration tools:
- Alembic (SQLAlchemy ecosystem)
- Django migrations
- custom migration scripts for small projects